## Encoding
### Opdracht

Zoals aan het begin van de cursus besproken gaat het project over handgeschreven nummers herkennen op een MysteryDevice met de volgende eigenschappen:

- Input scherm waarmee een nieuwe plaatje als ndarray aangemaakt kan worden door de gebruiker.
- Zeer beperkt RAM (256 KB)
- Beperkte opslag (1 MB voor programma + model)
- Geen GPU
- Embedded python

## Context

MysteryDevice opslaglimiet: 1 MB  
Volledige MNIST dataset: 52 MB

De encoding is nodig omdat de gebruiker een afbeelding invoert op het apparaat. We moeten die afbeelding zo compact mogelijk maken voor inference.

**Hoeveel RAM gebruikt de inputrepresentatie?**
- 784 bytes (1 byte per pixel, uint8)
- 32-bits floats: 784 x 4 = 3136 bytes = 3 KB

## Setup: MNIST laden (zonder sklearn)

In [ ]:
import numpy as np
import tensorflow as tf

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
# X_train shape: (60000, 28, 28), dtype: uint8

# Neem één sample als voorbeeld
sample = X_train[0]  # shape: (28, 28), dtype: uint8
print(f'Shape: {sample.shape}, dtype: {sample.dtype}')
print(f'Geheugen origineel: {sample.nbytes} bytes')

---
## Stap 1: Verkenning van encoding-methoden

In [ ]:
# --- Methode 1: Downscaling (28x28 -> 14x14) ---
# Elk 2x2 blokje wordt samengevoegd tot 1 pixel (gemiddelde).
# Voordeel: 4x minder pixels. Nadeel: fijne details gaan verloren.

def downscale(img):
    return img.reshape(14, 2, 14, 2).mean(axis=(1, 3)).astype(np.uint8)

ds = downscale(sample)
print(f'Downscaling (14x14): {ds.nbytes} bytes')

In [ ]:
# --- Methode 2: Binary thresholding ---
# Elke pixel wordt 0 (licht) of 1 (donker).
# Met np.packbits passen 8 pixels in 1 byte -> 784 pixels = 98 bytes.

def binary_threshold(img, threshold=128):
    binary = (img > threshold).astype(np.uint8)
    return np.packbits(binary.flatten())

bt = binary_threshold(sample)
print(f'Binary threshold (packed): {bt.nbytes} bytes')

In [ ]:
# --- Methode 3: 4-bit quantization ---
# Van 256 naar 16 grijsniveaus. Twee pixels per byte opslaan.

def quantize_4bit(img):
    q = (img // 16).astype(np.uint8)  # waarden 0-15
    flat = q.flatten()
    return (flat[0::2] << 4) | flat[1::2]  # 2 pixels per byte

q4 = quantize_4bit(sample)
print(f'4-bit quantization (packed): {q4.nbytes} bytes')

---
## Stap 2: Eigen encoding — Regio-feature extractie

### Idee
We verdelen de 28x28 afbeelding in een **4x4 raster** van blokken (elk 7x7 pixels).  
Per blok slaan we op: **hoeveel procent van de pixels donker is** (waarde > 50).  

Dit geeft **16 getallen -> 16 bytes** totaal.

### Waarom werkt dit?
Cijfers hebben herkenbare patronen in welke regio donker is:
- **'1'** -> inkt alleen in het midden
- **'0'** -> inkt aan de randen, leeg midden
- **'7'** -> inkt boven en rechtsonder

Deze ruimtelijke informatie blijft behouden met maar 16 features.

In [ ]:
def encode(img):
    """
    Eigen encoding: verdeel de 28x28 afbeelding in een 4x4 raster.
    Per blok (7x7 pixels): percentage donkere pixels (waarde > 50).
    Resultaat: 16 waarden als uint8 -> 16 bytes.
    """
    img = img.reshape(28, 28)
    features = []
    for rij in range(4):
        for kolom in range(4):
            blok = img[rij*7:(rij+1)*7, kolom*7:(kolom+1)*7]
            pct_donker = np.mean(blok > 50) * 100
            features.append(pct_donker)
    return np.array(features, dtype=np.uint8)  # 16 bytes

encoded_sample = encode(sample)
print(f'Encoded sample: {encoded_sample}')
print()
print(f'# bepaal hoeveel het gebruikt')
print(f'encoded_sample.nbytes = {encoded_sample.nbytes}')

---
## Stap 3: Vergelijking van alle methoden

In [ ]:
methoden = {
    'Origineel (28x28, uint8)':        sample.nbytes,
    'Downscaling (14x14, uint8)':      downscale(sample).nbytes,
    'Binary threshold (packed bits)':  binary_threshold(sample).nbytes,
    '4-bit quantization (packed)':     quantize_4bit(sample).nbytes,
    "Eigen encoding (4x4 regio's)":    encode(sample).nbytes,
}

print(f"{'Methode':<40} {'Bytes':>8} {'% van origineel':>16}")
print('-' * 68)
for naam, b in methoden.items():
    print(f"{naam:<40} {b:>8} bytes   {100*b/sample.nbytes:>6.1f}%")

---
## Stap 4: Beantwoording van de vragen

In [ ]:
print('=== Hoeveel RAM kost één afbeelding? ===')
print(f'  Origineel:      {sample.nbytes} bytes')
print(f'  Onze encoding:  {encode(sample).nbytes} bytes')
print()

print('=== Hoeveel RAM kost 100 afbeeldingen? ===')
print(f'  Origineel:      {100 * sample.nbytes} bytes = {100 * sample.nbytes / 1024:.1f} KB')
print(f'  Onze encoding:  {100 * encode(sample).nbytes} bytes = {100 * encode(sample).nbytes / 1024:.2f} KB')
print()

print('=== Hoe groot mag het model maximaal zijn? ===')
opslaglimiet = 1 * 1024 * 1024  # 1 MB in bytes
programma    = 50 * 1024        # schatting: ~50 KB voor Python-code
print(f'  Opslaglimiet:   {opslaglimiet // 1024} KB')
print(f'  Programma:     ~{programma // 1024} KB (schatting)')
print(f'  Max model:     ~{(opslaglimiet - programma) // 1024} KB')
print()

print('=== Kun je de encoding verder comprimeren? ===')
print('  Ja: de 16 waarden zijn 0-100, dat past in 7 bits.')
print('  Met 4-bit packing van de features -> 8 bytes per afbeelding.')
extra = (encode(sample) // 16).astype(np.uint8)
packed = (extra[0::2] << 4) | extra[1::2]
print(f'  4-bit gepackte features: {packed.nbytes} bytes')
print()

print('=== Is er informatie verloren gegaan? ===')
print('  Ja: exacte pixelwaarden en fijne details zijn weg.')
print('  De ruimtelijke structuur van het cijfer blijft behouden.')
print()

print('=== Kun je nog steeds het cijfer herkennen? ===')
print('  Ja. De 16 regio-features vangen het patroon van elk cijfer goed genoeg.')
print('  Een Decision Tree kan hierop getraind worden.')

---
## Conclusie

| Vraag | Antwoord |
|---|---|
| RAM voor 1 afbeelding | **16 bytes** (onze encoding) vs. 784 bytes origineel |
| RAM voor 100 afbeeldingen | **1600 bytes = 1.6 KB** |
| Max modelgrootte | **~974 KB** |
| Verdere compressie mogelijk? | Ja, 4-bit packing -> **8 bytes** per afbeelding |
| Informatieverlies? | Ja: exacte pixels en fijne details |
| Cijfer nog herkenbaar? | Ja |

### Onze keuzes:
- Geen normalisatie — we werken met een binaire drempel (> 50 = donker)
- Geen flattening — we werken in 2D blokken
- Feature extractie als encoding: 16 regio-intensiteiten
- Niet alle 784 pixels — slechts 16 samengevoegde features

Deze encoding is **49x kleiner** dan het origineel, heeft minimaal RAM nodig, weinig rekenkracht (alleen gemiddeldes), en behoudt genoeg informatie voor cijferherkenning.